# Lab 20 — Multi-Agent Research Demo Notebook

Notebook này giúp bạn **thử nghiệm nhanh** các khối logic của bài lab trước khi implement chính thức trong `src/`.

**Luồng làm việc:**
1. Khám phá schemas & shared state
2. Mock services (LLM + Search) để chạy không cần API key
3. Viết các agent demo (Researcher → Analyst → Writer)
4. Supervisor routing + vòng lặp workflow mini
5. Benchmark single-agent vs multi-agent

> ⚠️ **Quy tắc:** Notebook chỉ để prototype. Sau khi chạy được ở đây, bạn phải **chuyển logic vào `src/multi_agent_research_lab/`** và pass tests.

## 0. Setup

Chạy từ repo root với package đã cài (`pip install -e ".[dev]"`).

In [ ]:
import sys
from pathlib import Path

# Cho phép import package khi chạy notebook từ thư mục notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

from multi_agent_research_lab.core.schemas import (
    AgentName,
    AgentResult,
    BenchmarkMetrics,
    ResearchQuery,
    SourceDocument,
)
from multi_agent_research_lab.core.state import ResearchState

print("✅ Import OK — package sẵn sàng")

## 1. Khám phá Shared State

`ResearchState` là **single source of truth** được truyền qua mọi agent. Mỗi agent đọc state, cập nhật, rồi trả lại.

In [ ]:
query = ResearchQuery(
    query="So sánh RAG và fine-tuning cho domain adaptation",
    max_sources=3,
)
state = ResearchState(request=query)

state.record_route("researcher")
state.add_trace_event("demo", {"note": "first route recorded"})

print("Iteration:", state.iteration)
print("Route history:", state.route_history)
print("Trace:", state.trace)

## 2. Mock Services

Để demo không cần API key, ta dùng mock. Trong bản chính thức (`src/services/`), bạn sẽ nối provider thật (OpenAI / Tavily...).

- `MockSearchClient`: **đã viết sẵn** làm mẫu.
- `MockLLMClient`: Giả lập phản hồi LLM theo vai trò prompt.

In [ ]:
from dataclasses import dataclass


class MockSearchClient:
    """Trả về nguồn giả lập — cùng interface với services.search_client.SearchClient."""

    _FAKE_DOCS = [
        SourceDocument(
            title="RAG vs Fine-tuning: A Practical Guide",
            url="https://example.com/rag-vs-ft",
            snippet="RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.",
        ),
        SourceDocument(
            title="Retrieval-Augmented Generation Survey",
            url="https://example.com/rag-survey",
            snippet="RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.",
        ),
        SourceDocument(
            title="When to Fine-tune LLMs",
            url="https://example.com/when-finetune",
            snippet="Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.",
        ),
    ]

    def search(self, query: str, max_results: int = 5) -> list[SourceDocument]:
        return self._FAKE_DOCS[:max_results]


@dataclass(frozen=True)
class MockLLMResponse:
    content: str
    input_tokens: int | None = None
    output_tokens: int | None = None


class MockLLMClient:
    """Giả lập LLM — cùng interface với services.llm_client.LLMClient."""

    def complete(self, system_prompt: str, user_prompt: str) -> MockLLMResponse:
        sys_lower = system_prompt.lower()
        if "analyst" in sys_lower:
            content = (
                "1. RAG tối ưu cho tri thức biến động; Fine-tuning tối ưu cho format/style.\n"
                "2. Nguồn tài liệu uy tín, độ bao phủ tốt các khía cạnh kỹ thuật."
            )
        elif "writer" in sys_lower:
            content = (
                "# So Sánh RAG và Fine-tuning\n\n"
                "## Tổng quan\n"
                "RAG giảm thiểu hallucination hiệu quả [1]. Fine-tuning định hình phong cách phản hồi [2].\n\n"
                "## Tài liệu tham khảo\n"
                "[1] RAG vs Fine-tuning: A Practical Guide (https://example.com/rag-vs-ft)\n"
                "[2] Retrieval-Augmented Generation Survey (https://example.com/rag-survey)"
            )
        else:
            content = f"Baseline Response for: {user_prompt}"

        in_tok = max(1, (len(system_prompt) + len(user_prompt)) // 4)
        out_tok = max(1, len(content) // 4)
        return MockLLMResponse(content=content, input_tokens=in_tok, output_tokens=out_tok)


# Smoke test phần đã cho sẵn
search_client = MockSearchClient()
docs = search_client.search(query.query, max_results=query.max_sources)
for d in docs:
    print(f"- {d.title}: {d.snippet[:60]}...")

## 3. Demo Agents

Mỗi agent tuân theo contract `BaseAgent.run(state) -> state`.

- `DemoResearcherAgent`: thu thập nguồn và ghi `sources` + `research_notes`.
- `DemoAnalystAgent`: tổng hợp `sources` thành `analysis_notes`.
- `DemoWriterAgent`: viết `final_answer` kèm citation.

In [ ]:
class DemoResearcherAgent:
    """MẪU: thu thập nguồn và ghi chú nghiên cứu."""

    name = "researcher"

    def __init__(self, search_client: MockSearchClient) -> None:
        self.search_client = search_client

    def run(self, state: ResearchState) -> ResearchState:
        docs = self.search_client.search(
            state.request.query, max_results=state.request.max_sources
        )
        state.sources = docs
        state.research_notes = "\n".join(f"- {d.title}: {d.snippet}" for d in docs)
        state.agent_results.append(
            AgentResult(
                agent=AgentName.RESEARCHER,
                content=state.research_notes,
                metadata={"num_sources": len(docs)},
            )
        )
        state.add_trace_event("researcher.done", {"num_sources": len(docs)})
        return state


class DemoAnalystAgent:
    """Phân tích sources thành analysis_notes."""

    name = "analyst"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        if not state.sources:
            state.errors.append("no sources to analyze")
            return state
        resp = self.llm_client.complete("You are an analyst.", state.research_notes or "")
        state.analysis_notes = resp.content
        state.agent_results.append(
            AgentResult(agent=AgentName.ANALYST, content=resp.content)
        )
        state.add_trace_event("analyst.done", {"length": len(resp.content)})
        return state


class DemoWriterAgent:
    """Viết final_answer có trích dẫn nguồn."""

    name = "writer"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        context = state.analysis_notes or state.research_notes or ""
        resp = self.llm_client.complete("You are a writer.", context)
        state.final_answer = resp.content
        state.agent_results.append(
            AgentResult(agent=AgentName.WRITER, content=resp.content)
        )
        state.add_trace_event("writer.done", {"sources_cited": len(state.sources)})
        return state


# Smoke test agent mẫu
state = ResearchState(request=query)
state = DemoResearcherAgent(search_client).run(state)
print(state.research_notes)

## 4. Supervisor Routing

Supervisor quyết định agent nào chạy tiếp dựa trên state hiện tại. Đây là **trái tim của bài lab** — bạn tự thiết kế policy.

In [ ]:
MAX_ITERATIONS = 6


def demo_supervisor_route(state: ResearchState) -> str:
    """Trả về một trong: 'researcher' | 'analyst' | 'writer' | 'done'."""
    if state.iteration >= MAX_ITERATIONS:
        return "done"
    if not state.sources:
        return "researcher"
    if not state.analysis_notes:
        return "analyst"
    if not state.final_answer:
        return "writer"
    return "done"

## 5. Mini Workflow Loop

Vòng lặp điều phối workflow mini mô phỏng các node và edges.

In [ ]:
def run_demo_workflow(query_text: str) -> ResearchState:
    q = ResearchQuery(query=query_text, max_sources=3)
    state = ResearchState(request=q)

    llm = MockLLMClient()
    agents = {
        "researcher": DemoResearcherAgent(MockSearchClient()),
        "analyst": DemoAnalystAgent(llm),
        "writer": DemoWriterAgent(llm),
    }

    while True:
        route = demo_supervisor_route(state)
        state.record_route(route)
        if route == "done":
            break
        state = agents[route].run(state)

    return state


final_state = run_demo_workflow("So sánh RAG và fine-tuning cho domain adaptation")
print("Route history:", final_state.route_history)
print("\n=== FINAL ANSWER ===\n")
print(final_state.final_answer)

## 6. Benchmark: Single-agent vs Multi-agent

Dùng `run_benchmark` từ package để so sánh.

In [ ]:
from multi_agent_research_lab.evaluation.benchmark import run_benchmark


def run_single_agent(query_text: str) -> ResearchState:
    """Baseline: một lần gọi LLM duy nhất, không search, không phân tích."""
    q = ResearchQuery(query=query_text)
    st = ResearchState(request=q)
    resp = MockLLMClient().complete("You are a general AI.", query_text)
    st.final_answer = resp.content
    st.iteration = 1
    return st


def compute_citation_coverage(state: ResearchState) -> float:
    """Tỷ lệ nguồn trong state.sources được nhắc đến trong final_answer."""
    if not state.sources or not state.final_answer:
        return 0.0
    cited = 0
    for i, s in enumerate(state.sources):
        if f"[{i+1}]" in state.final_answer or s.title in state.final_answer:
            cited += 1
    return cited / len(state.sources)


demo_query = "So sánh RAG và fine-tuning cho domain adaptation"

results: list[BenchmarkMetrics] = []
for run_name, runner in [
    ("single_agent", run_single_agent),
    ("multi_agent", run_demo_workflow),
]:
    st, metrics = run_benchmark(run_name, demo_query, runner)
    metrics.citation_coverage = compute_citation_coverage(st)
    results.append(metrics)

print(f"{'run':<15}{'latency (s)':<15}{'citation cov.':<15}")
for m in results:
    print(f"{m.run_name:<15}{m.latency_seconds:<15.3f}{m.citation_coverage!s:<15}")

## 7. Next Steps — chuyển sang `src/`

Khi notebook chạy end-to-end, chuyển logic vào code chính thức:

| Notebook | Đích trong `src/multi_agent_research_lab/` |
|---|---|
| `MockLLMClient` → provider thật | `services/llm_client.py` |
| `MockSearchClient` → provider thật | `services/search_client.py` |
| `DemoResearcherAgent` / `DemoAnalystAgent` / `DemoWriterAgent` | `agents/researcher.py`, `agents/analyst.py`, `agents/writer.py` |
| `demo_supervisor_route` | `agents/supervisor.py` |
| `run_demo_workflow` → LangGraph nodes/edges | `graph/workflow.py` |
| `compute_citation_coverage` + quality score | `evaluation/benchmark.py` |

Sau đó verify:
```bash
make lint && make test
python -m multi_agent_research_lab.cli run --query "..."
bash scripts/check_todos.sh   # đảm bảo không còn TODO trong src/
```